# Notebook 16 — Limitations and Next Steps

## Purpose
An honest documentation of what this project does NOT do, where the analysis
may be misleading, and what would be needed to make it more rigorous.

## Why this matters
Stating limitations is not weakness — it is scientific rigour.  It signals
to reviewers and hiring managers that I know the boundaries of my own work.


In [ ]:
# This notebook is mostly markdown documentation.
# I run a quick check to confirm all expected output files exist.

import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths

cfg   = load_config()
paths = Paths(cfg)

expected_outputs = [
    paths.processed / cfg['data']['master_file'],
    paths.processed / cfg['data']['rfm_file'],
    paths.processed / 'train.parquet',
    paths.processed / 'test.parquet',
    paths.models / 'rf_review_model.joblib',
    paths.models / 'xgb_review_model.joblib',
    paths.models / 'advanced_results.json',
]

print("=== Output artifact check ===")
all_ok = True
for p in expected_outputs:
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status:8s}  {p.name}")
    if status == "MISSING":
        all_ok = False

if all_ok:
    print("\nAll expected outputs present. Project pipeline complete.")
else:
    print("\nSome outputs missing. Check which notebook failed.")


## Key Limitations

### 1. Prediction accuracy is modest

The Random Forest achieves ~[X]% accuracy on 5-class review score prediction
(baseline ~55%).  This is because:

- Review scores reflect many unmeasured factors (product quality, packaging,
  seller communication quality, product photos).
- The class distribution is heavily skewed toward 5-star reviews.
- We use integer encoding for state and category, which loses geographic
  and semantic structure.

**What would improve it:** product description NLP, seller-level features,
customer tenure features.

### 2. RFM is descriptive, not causal

Segment labels ("Champions", "At Risk") are convenient shorthand, not
proven causal states.  A "Champion" customer might stop buying for reasons
unrelated to their RFM score (product no longer available, moved abroad, etc.).

**What would improve it:** survival analysis, causal uplift modelling with A/B tests.

### 3. Geographic features are coarse

Customer and seller state is the geographic feature.  Real delivery time depends
on exact origin-destination distance, road network, and carrier routing.

**What would improve it:** geolocation lookup (already in the dataset) to compute
haversine distance between seller and customer ZIP codes.

### 4. Time-based split may understate generalisation error

The test set covers the last months of the dataset.  Seasonal patterns in that
period may not be representative of other years.

**What would improve it:** rolling-window cross-validation.

## Next Steps (priority order)

1. **NLP on review comments** — extract sentiment features, specific complaint topics.
2. **Distance feature** — compute seller-to-customer distance from geolocation table.
3. **Seller-level features** — seller tenure, average historical review, return rate.
4. **CLV modelling** — Pareto/NBD model for customer lifetime value.
5. **Hyperparameter tuning** — RandomizedSearchCV with nested CV.
6. **Deployment** — wrap the review predictor in a FastAPI endpoint.
7. **Causal inference** — estimate treatment effect of free shipping.


In [ ]:
print("Notebook 16 complete.")
print()
print("Project summary:")
print("  - 16 analytical notebooks completed")
print("  - 1 Streamlit dashboard available at website_or_demo/app.py")
print("  - Full report in paper_or_report/report.md")
print("  - Tests: pytest tests/")
print()
print("To launch dashboard: streamlit run website_or_demo/app.py")
